# Predicting Student Exam Scores with Feature Engineering
## Practice Skeleton

**Short name (GitHub):** `ExamPred`

**Adaptation of:** `LaptopPrice` / `TaxCreate` / `LoanOrig` (unit-bearing strings → numeric features → encode → Random Forest).

**Card:** `data/student_exams.csv` — **920** course records × **24** columns. Target `exam_score` is 0–100 (mean ≈ 73.0, median 72.9). Hours, GPA, attendance, credits, partners, term, stars, and cohort arrive as text (`"12.0 hours"`, `"3.40 GPA"`, `"91.0%"`, `"15 cr"`, `"2 partners"`, `"16-week"`, `"4 stars"`, `"2023 CY"`).

**How to use**
- Fill cells marked `# YOUR CODE HERE`. Keep the cheat-sheet and flowchart visible.
- Compare with `ExamPred_Solution.ipynb` only after an attempt.
- Data: `data/student_exams.csv`. Charts: `exampred_*.png`.
- Clone with `ExamPred_Reusable_Template.ipynb`.
- **Not a placement exam, not an admissions model, not a grading engine.** Teaching roster only.


## Inline cheat-sheet (keep this cell visible)

See also **`ExamPred_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Hours | `.str.replace(' hours','').astype(float)` |
| GPA | strip `' GPA'` |
| Percent | strip `'%'` then `/ 100` |
| Composite | `study_load = study_hours + prep_hours` |
| Partners / credits / term | extract digits; strip `' cr'`, `'-week'` |
| Sentinel cohort | `'Not Available'` → `'0'` *before* stripping `' CY'` |
| Cardinality | `< 5` → one-hot. `≥ 5` → train-only mean `exam_score` |
| Leak | `effort_stars` tracks the score (corr ≈ 0.94). Drop it for a *pre-exam* model. |
| Split / model | `test_size=0.2`, `random_state=42` / `RandomForestRegressor(random_state=42)` |
| Metrics | MAE in score points + R² vs the **mean-score baseline** |
| Never | Use this to assign a real grade or admit/deny a student. |


## Flowchart of the desired outcome

![ExamPred flow](exampred_flowchart.png)

Load the roster → strip hours / GPA / % / cr → engineer `study_load` → clean stars, partners, term, cohort → EDA → encode → 80/20 → forest vs linear / mean → then **drop `effort_stars`** and refit the pre-exam card.


## 0. Packages


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
sns.set_theme(style="whitegrid")


## 1. Load and inspect

Print `.head()`, `.info()`, `.shape()`, `.nunique()`, `df['exam_score'].describe()`.


In [ ]:
# YOUR CODE HERE
df = None


## 2. Parse hours / GPA and engineer study load

Loop over `['study_hours','prep_hours','sleep_hours']`, strip `" hours"`.
Strip `" GPA"` from `prior_gpa`. Create `study_load = study_hours + prep_hours`.


In [ ]:
# YOUR CODE HERE


## 3. Clean rate, stars, cohort, partners, term, credits

- `attendance`: strip `%` → float / 100.
- `effort_stars`: strip `" stars"` then `" star"` → int.
- `cohort`: `"Not Available"` → `"0"`, strip `" CY"` → `cohort_n`, drop original.
- `study_partners`: extract digits → int.
- `term_weeks`: strip `"-week"` → int.
- `credits`: strip `" cr"` → int.


In [ ]:
# YOUR CODE HERE


## 4. EDA

Heatmap; partners box; effort-star box; score histogram.

![heatmap](exampred_heatmap.png)
![partners](exampred_partners_box.png)
![stars](exampred_stars_box.png)


In [ ]:
# YOUR CODE HERE
numeric_df = None


## 5. Encode + split

High-card: `campus`, `subject`, `standing`. Low-card: delivery, course_type, complexity, timing, digital_hw, tutoring, resident, section_size.


In [ ]:
# YOUR CODE HERE
X_train = X_test = y_train = y_test = None


## 6. Fit the Random Forest


In [ ]:
# YOUR CODE HERE
rf_model = None
y_pred = None


## 7. Evaluate

Print MAE, RMSE, R², mean-score baseline.

![importance](exampred_importance.png)
![pred vs actual](exampred_pred_actual.png)


In [ ]:
# YOUR CODE HERE
mae = r2 = None


## 8. Alternate code


### 8a. Regex extract


In [ ]:
# YOUR CODE HERE


### 8b. Leakage-safe encoding


In [ ]:
# YOUR CODE HERE


### 8c. Linear / Ridge


In [ ]:
# YOUR CODE HERE


### 8d. Permutation importance


In [ ]:
# YOUR CODE HERE


## 9. More practice

1. **Drop `effort_stars`** (pre-exam card). Expect MAE ≈ 5.1 and R² ≈ 0.54.
2. Pass flag at `exam_score >= 60`; report accuracy of `y_pred >= 60`.
3. Fit only `subject == 'Algebra'`.
4. Tail MAE on scores < 60.
5. Hours-only formula: `50 + 1.4*study_hours + 8*(prior_gpa-2.5)` vs the forest.


In [ ]:
# YOUR CODE HERE — pick at least two


## 10. Simulation

![simulation](exampred_simulation.png)

Reference with stars: MAE ≈ **2.33**, R² ≈ **0.912**. Pre-exam (no stars): MAE ≈ **5.10**, R² ≈ **0.541**.


In [ ]:
N_EST = 100
MAX_DEPTH = None
NOISE_SD = 0
SUBSAMPLE = 1.0
RANDOM_STATE = 42


In [ ]:
# YOUR CODE HERE


## 11. What this model can and cannot do

**Can**
- Reconstruct a noisy linear score rule from parsed hours, GPA, and attendance.
- Beat a mean-score guess (2.3 points with stars, 5.1 without, vs 7.7 baseline).
- Show that `effort_stars` is a band of the target, not a study habit you can assign before the exam.

**Cannot**
- Grade a live exam or place a student in a course.
- Use assignment *counts* as if they were known on day one of the term.
- Transfer to another campus or year without a refresh.

**Same pipeline:** midterm → final, homework-points → course mark, certification exam scores, training-assessment batteries.
